## Tasks

### Task 1: Build a Reusable `bootstrap_ci` Function

Write a function with the following signature:

```python
def bootstrap_ci(data, stat_func=np.mean, n_boot=10_000, ci_level=0.95):
    """
    Returns (lower, upper) percentile bootstrap CI.

    Parameters
    ----------
    data : array-like
        The sample to resample from.
    stat_func : callable
        The statistic to compute on each resample (default: np.mean).
    n_boot : int
        Number of bootstrap resamples.
    ci_level : float
        Confidence level (e.g. 0.95 for a 95 % CI).
    """
```

**Steps:**

1. Inside the function, use `np.random.choice` (with replacement) to draw `n_boot` resamples of the same size as `data`.
2. Compute `stat_func` on each resample and store the results in an array called `boot_stats`.
3. Use `np.percentile` to extract the lower and upper bounds based on `ci_level`. For a 95 % CI these are the 2.5th and 97.5th percentiles.
4. Return `(lower, upper)`.

**Validation:** Call your function on `np.arange(1, 101)` with the default mean. The 95 % CI should be roughly (40, 60). Verify this before moving on.

In [21]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

In [22]:
def bootstrap_ci(data, stat_func=np.mean, n_boot=10_000, ci_level=0.95):
    """
    Returns (lower, upper) percentile bootstrap CI.

    Parameters
    ----------
    data : array-like
        The sample to resample from.
    stat_func : callable
        The statistic to compute on each resample (default: np.mean).
    n_boot : int
        Number of bootstrap resamples.
    ci_level : float
        Confidence level (e.g. 0.95 for a 95 % CI).
    """

    data = np.array(data)  
    n = len(data)    
    
    boot_stats = []     

    for _ in range(n_boot):
        sample = np.random.choice(data, size=n, replace=True)
        stat = stat_func(sample)
        boot_stats.append(stat)
    
    boot_stats = np.array(boot_stats)
    
    alpha = 1 - ci_level
    lower = np.percentile(boot_stats, 100 * (alpha / 2))
    upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    
    return lower, upper

data = np.arange(1, 101)

ci = bootstrap_ci(data)

print(ci)

(np.float64(44.799749999999996), np.float64(56.100249999999996))


### Task 2: Apply Bootstrap CIs to Real Data

Load or generate a dataset with at least 200 rows that includes:

- A **continuous numeric** column (e.g. income, test score, response time)
- A **binary / boolean** column (e.g. purchased yes/no, passed yes/no)

You may use any publicly available dataset or generate synthetic data with `np.random.normal` / `np.random.binomial`.

**Compute the following bootstrap CIs (95 %, 10 000 resamples each):**

| # | Statistic | Column | `stat_func` |
|---|-----------|--------|-------------|
| 1 | Mean | Continuous column | `np.mean` |
| 2 | Median | Continuous column | `np.median` |
| 3 | Proportion | Binary column | `np.mean` (on 0/1 values) |

Present the three intervals in a tidy summary table using `pd.DataFrame`.


In [23]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 300

income = np.random.normal(loc=5000, scale=1000, size=n)

purchased = np.random.binomial(n=1, p=0.7, size=n)

df = pd.DataFrame({
    "income": income,
    "purchased": purchased
})

df.head()

,income,purchased
0,5496.714153,1
1,4861.735699,1
2,5647.688538,0
3,6523.029856,0
4,4765.846625,0


In [24]:
mean_ci = bootstrap_ci(df["income"], stat_func=np.mean)

median_ci = bootstrap_ci(df["income"], stat_func=np.median)

prop_ci = bootstrap_ci(df["purchased"], stat_func=np.mean)


summary = pd.DataFrame({
    "Statistic": ["Mean", "Median", "Proportion"],
    "Column": ["income", "income", "purchased"],
    "CI Lower": [mean_ci[0], median_ci[0], prop_ci[0]],
    "CI Upper": [mean_ci[1], median_ci[1], prop_ci[1]]
})

summary

,Statistic,Column,CI Lower,CI Upper
0,Mean,income,4883.781073,5105.625104
1,Median,income,4854.975302,5178.907392
2,Proportion,purchased,0.640000,0.746667


### Task 3: Compare Bootstrap CIs with Normal-Approximation CIs

For the **mean** and the **proportion** from Task 2, compute classical normal-approximation 95 % CIs:

- **Mean CI:** Use `st.t.interval` (or `st.norm.interval`) with the sample mean and standard error.
- **Proportion CI:** Use the Wald interval formula: p̂ ± z × √(p̂(1 − p̂) / n), where z = 1.96.

Create a comparison table with columns: `Statistic`, `Bootstrap Lower`, `Bootstrap Upper`, `Normal Lower`, `Normal Upper`.

**Answer in a Markdown cell:**

1. Are the two approaches giving similar intervals? Where do they diverge, if at all?
2. For which statistic (mean vs. proportion vs. median) is the bootstrap approach especially useful, and why?

In [25]:
import scipy.stats as st

mean = df["income"].mean()
se = df["income"].std(ddof=1) / np.sqrt(len(df))

normal_ci_mean = st.t.interval(0.95, df=len(df)-1, loc=mean, scale=se)


p_hat = df["purchased"].mean()
n = len(df)
z = 1.96

se_p = np.sqrt(p_hat * (1 - p_hat) / n)
normal_ci_prop = (p_hat - z*se_p, p_hat + z*se_p)

In [26]:
comparison = pd.DataFrame({
    "Statistic": ["Mean", "Proportion"],
    "Bootstrap Lower": [mean_ci[0], prop_ci[0]],
    "Bootstrap Upper": [mean_ci[1], prop_ci[1]],
    "Normal Lower": [normal_ci_mean[0], normal_ci_prop[0]],
    "Normal Upper": [normal_ci_mean[1], normal_ci_prop[1]]
})

comparison

,Statistic,Bootstrap Lower,Bootstrap Upper,Normal Lower,Normal Upper
0,Mean,4883.781073,5105.625104,4882.628857,5106.274024
1,Proportion,0.640000,0.746667,0.641154,0.745513


### Are the two approaches giving similar intervals? Where do they diverge, if at all?

Yes, for both the mean and the proportion, the bootstrap and normal-approximation confidence intervals are very similar. The differences in the lower and upper bounds are minimal, almost negligible in practice.

They may diverge in situations where:

The sample size is small
The data is skewed or has outliers
The statistic of interest is not the mean or proportion (e.g., median or quantiles)

In such cases, the bootstrap CI tends to be more reliable because it does not rely on assumptions about the data distribution.


### For which statistic (mean vs. proportion vs. median) is the bootstrap approach especially useful, and why?


The bootstrap approach is especially useful for the median (and other non-standard statistics like quantiles or custom metrics).

Reason: The sampling distribution of the median is not easy to derive analytically, so normal-approximation formulas are not straightforward.
Bootstrap works by resampling the actual data many times, making it robust to skewed distributions or outliers.

For mean and proportion, normal approximation usually works well when the sample size is large, but bootstrap provides a data-driven, assumption-free estimate.